In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "gpu")
print("Device:", device)

Device: cuda


In [ ]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914,0.4822,0.4465),
        (0.2023,0.1994,0.2010)
    )
])

def preprocess(image):
    img = Image.open(image).convert("RGB")
    return transform(img).unsqueeze(0)

In [5]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = train_dataset.classes
print(classes)

100%|██████████| 170M/170M [00:03<00:00, 42.7MB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
import torch.nn as nn

if __name__ == "__main__":
    class CNN(nn.Module):
        def __init__(self):
            super(CNN, self).__init__()

            self.features = nn.Sequential(
                nn.Conv2d(3,64,3,padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),

                nn.Conv2d(64,64,3,padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.MaxPool2d(2),

                nn.Conv2d(64,128,3,padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),

                nn.Conv2d(128,128,3,padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.MaxPool2d(2),

                nn.Conv2d(128,256,3,padding=1),
                nn.BatchNorm2d(256),
                nn.ReLU(),

                nn.Conv2d(256,256,3,padding=1),
                nn.BatchNorm2d(256),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )

            self.classifier = nn.Sequential(
                nn.Linear(256*4*4,512),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(512,10)
            )

        def forward(self,x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
            x = self.classifier(x)
            return x

In [7]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                            step_size=20,
                                            gamma=0.1)

In [8]:
num_epochs = 30

train_acc_list = []
train_loss_list = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_loss_list.append(epoch_loss)
    train_acc_list.append(epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Loss {epoch_loss:.4f} | Acc {epoch_acc:.2f}%")

Epoch 1/30 | Loss 1.7810 | Acc 33.49%
Epoch 2/30 | Loss 1.3121 | Acc 52.12%
Epoch 3/30 | Loss 1.0807 | Acc 61.62%
Epoch 4/30 | Loss 0.9529 | Acc 66.77%
Epoch 5/30 | Loss 0.8518 | Acc 70.47%
Epoch 6/30 | Loss 0.7727 | Acc 73.86%
Epoch 7/30 | Loss 0.7018 | Acc 76.34%
Epoch 8/30 | Loss 0.6576 | Acc 78.01%
Epoch 9/30 | Loss 0.6045 | Acc 79.93%
Epoch 10/30 | Loss 0.5626 | Acc 81.36%
Epoch 11/30 | Loss 0.5280 | Acc 82.52%
Epoch 12/30 | Loss 0.4907 | Acc 84.04%
Epoch 13/30 | Loss 0.4718 | Acc 84.61%
Epoch 14/30 | Loss 0.4470 | Acc 85.57%
Epoch 15/30 | Loss 0.4219 | Acc 86.35%
Epoch 16/30 | Loss 0.3993 | Acc 87.04%
Epoch 17/30 | Loss 0.3775 | Acc 87.66%
Epoch 18/30 | Loss 0.3579 | Acc 88.19%
Epoch 19/30 | Loss 0.3406 | Acc 88.72%
Epoch 20/30 | Loss 0.3168 | Acc 89.66%
Epoch 21/30 | Loss 0.2485 | Acc 91.92%
Epoch 22/30 | Loss 0.2195 | Acc 92.94%
Epoch 23/30 | Loss 0.2107 | Acc 93.05%
Epoch 24/30 | Loss 0.2047 | Acc 93.23%
Epoch 25/30 | Loss 0.1968 | Acc 93.52%
Epoch 26/30 | Loss 0.1915 | Acc 93

In [9]:
torch.save(model.state_dict(), "model_CIFAR.pth")

In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print("Test Accuracy:", test_accuracy)

Test Accuracy: 90.62
